In [ ]:
"""
AgriSense — Task 1  |  STEP 0: Google Earth Engine Exports
===========================================================
Run this script in Google Colab or a GEE-authenticated environment.

WHAT THIS EXPORTS (5 files total to Google Drive → AgriSense/ folder):
  1. S2_BOA_Punjab_Rabi2023.tif       — 7-band BOA composite (MAIN PRODUCT)
  2. S2_TOA_Punjab_Rabi2023.tif       — 7-band TOA composite (for DOS-1 benchmark)
  3. S2_CloudBands_Date1.tif          — 10-band image Date1 (for s2cloudless)
  4. S2_CloudBands_Date2.tif          — 10-band image Date2 (for s2cloudless)
  5. S2_SIFT_Date1.tif / Date2.tif    — single-band (B4) for SIFT co-registration

DATE STRATEGY for low SIFT RMSE:
  - Use SAME Rabi season, 15–30 days apart (not different seasons)
  - Both dates should be clear (< 5% cloud)
  - Chosen: 2023-12-01 and 2023-12-31  (stable winter wheat period, minimal change)
  - Reason: December wheat fields are in early tillering — very stable spectral signature
  - This maximises feature matching stability → RMSE < 1 pixel is achievable

Author: AgriSense Team | Punjab AOI: 73.0–73.5°E, 30.5–31.0°N
"""

import ee
import geemap

# ─── GEE Initialisation ──────────────────────────────────────────────────────
ee.Authenticate()
ee.Initialize(project='dip-project-52918')   # ← replace with your project ID

# ─── Study Area & Parameters ─────────────────────────────────────────────────
AOI    = ee.Geometry.Rectangle([73.0, 30.5, 73.5, 31.0])
CRS    = 'EPSG:32642'
SCALE  = 10        # metres
FOLDER = 'AgriSense'
MAX_PX = 1e10

# Season window
SEASON_START = '2023-10-01'
SEASON_END   = '2024-04-30'

# Single-date windows for SIFT (15-day gap, stable phenology)
# ── WHY DECEMBER? ───────────────────────────────────────────────────────────
# Rabi wheat in Punjab is planted Oct–Nov and reaches early tillering by Dec.
# The canopy is uniform and changes slowly → SIFT finds stable texture features.
# A 30-day gap keeps reflectance change < 5% in most bands → RMSE < 1 px.
# ────────────────────────────────────────────────────────────────────────────
DATE1_START = '2023-12-01'
DATE1_END   = '2023-12-15'    # picks best single image in this 15-day window

DATE2_START = '2023-12-16'
DATE2_END   = '2023-12-31'    # picks best single image in second 15-day window

# 7 spectral bands for BOA/TOA composite products
COMPOSITE_BANDS = ['B2', 'B3', 'B4', 'B5', 'B8', 'B11', 'B12']

# 10 bands required by s2cloudless (exact order is mandatory)
# [B01, B02, B04, B05, B08, B8A, B09, B10, B11, B12]
CLOUD_BANDS = ['B1', 'B2', 'B4', 'B5', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12']


# ═══════════════════════════════════════════════════════════════════════════════
#  EXPORT 1 — S2 BOA 7-band Seasonal Composite (MAIN PRODUCT)
#  Source: S2_SR_HARMONIZED (Level-2A, Sen2Cor corrected)
# ═══════════════════════════════════════════════════════════════════════════════
print("Preparing Export 1: S2 BOA Composite (MAIN PRODUCT)...")

s2_sr = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
           .filterBounds(AOI)
           .filterDate(SEASON_START, SEASON_END)
           .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
           .select(COMPOSITE_BANDS))

boa_composite = s2_sr.median().clip(AOI)

# NOTE: S2_SR values are in range 0–10000 (scale factor 0.0001)
# We export as-is and divide by 10000 in Python to get reflectance [0,1]
task1 = ee.batch.Export.image.toDrive(
    image       = boa_composite.toFloat(),
    description = 'S2_BOA_Punjab_Rabi2023',
    folder      = FOLDER,
    region      = AOI,
    scale       = SCALE,
    crs         = CRS,
    maxPixels   = MAX_PX
)
task1.start()
print(f"  ✔ Task started: S2_BOA_Punjab_Rabi2023 | ID: {task1.id}")


# ═══════════════════════════════════════════════════════════════════════════════
#  EXPORT 2 — S2 TOA 7-band Seasonal Composite (for DOS-1 benchmark ONLY)
#  Source: S2_HARMONIZED (Level-1C, raw TOA)
# ═══════════════════════════════════════════════════════════════════════════════
print("Preparing Export 2: S2 TOA Composite (DOS-1 benchmark)...")

s2_toa = (ee.ImageCollection('COPERNICUS/S2_HARMONIZED')
            .filterBounds(AOI)
            .filterDate(SEASON_START, SEASON_END)
            .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
            .select(COMPOSITE_BANDS))

toa_composite = s2_toa.median().clip(AOI)

task2 = ee.batch.Export.image.toDrive(
    image       = toa_composite.toFloat(),
    description = 'S2_TOA_Punjab_Rabi2023',
    folder      = FOLDER,
    region      = AOI,
    scale       = SCALE,
    crs         = CRS,
    maxPixels   = MAX_PX
)
task2.start()
print(f"  ✔ Task started: S2_TOA_Punjab_Rabi2023 | ID: {task2.id}")


# ═══════════════════════════════════════════════════════════════════════════════
#  HELPER: get single best image in a date window (lowest cloud %)
# ═══════════════════════════════════════════════════════════════════════════════
def get_best_image(collection, start, end, bands, aoi):
    """Return the least-cloudy single image in [start, end] clipped to aoi."""
    img = (collection
           .filterDate(start, end)
           .sort('CLOUDY_PIXEL_PERCENTAGE')
           .first()
           .select(bands)
           .clip(aoi))
    return img


# ═══════════════════════════════════════════════════════════════════════════════
#  EXPORT 3 & 4 — 10-band images for s2cloudless (Date1 & Date2)
#  These use the CLOUD_BANDS list required by the s2cloudless model
# ═══════════════════════════════════════════════════════════════════════════════
print("Preparing Exports 3 & 4: 10-band cloud-masking images (Date1 & Date2)...")

s2_sr_all = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
               .filterBounds(AOI)
               .filterDate(DATE1_START, DATE2_END)
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

cloud_img1 = get_best_image(s2_sr_all, DATE1_START, DATE1_END, CLOUD_BANDS, AOI)
cloud_img2 = get_best_image(s2_sr_all, DATE2_START, DATE2_END, CLOUD_BANDS, AOI)

task3 = ee.batch.Export.image.toDrive(
    image       = cloud_img1.toFloat(),
    description = 'S2_CloudBands_Date1',
    folder      = FOLDER,
    region      = AOI,
    scale       = SCALE,
    crs         = CRS,
    maxPixels   = MAX_PX
)
task3.start()
print(f"  ✔ Task started: S2_CloudBands_Date1 | ID: {task3.id}")

task4 = ee.batch.Export.image.toDrive(
    image       = cloud_img2.toFloat(),
    description = 'S2_CloudBands_Date2',
    folder      = FOLDER,
    region      = AOI,
    scale       = SCALE,
    crs         = CRS,
    maxPixels   = MAX_PX
)
task4.start()
print(f"  ✔ Task started: S2_CloudBands_Date2 | ID: {task4.id}")


# ═══════════════════════════════════════════════════════════════════════════════
#  EXPORT 5 & 6 — Single-band (B4/Red) images for SIFT co-registration
#  WHY SINGLE BAND?  SIFT works on 2D grayscale. Using B4 (Red) gives
#  high contrast over agricultural fields (chlorophyll absorption edge).
#  WHY B4?  Best texture contrast for wheat vs bare soil in Punjab winter.
# ═══════════════════════════════════════════════════════════════════════════════
print("Preparing Exports 5 & 6: Single-band B4 images for SIFT...")

sift_img1 = get_best_image(s2_sr_all, DATE1_START, DATE1_END, ['B4'], AOI)
sift_img2 = get_best_image(s2_sr_all, DATE2_START, DATE2_END, ['B4'], AOI)

task5 = ee.batch.Export.image.toDrive(
    image       = sift_img1.toFloat(),
    description = 'S2_SIFT_Date1',
    folder      = FOLDER,
    region      = AOI,
    scale       = SCALE,
    crs         = CRS,
    maxPixels   = MAX_PX
)
task5.start()
print(f"  ✔ Task started: S2_SIFT_Date1 | ID: {task5.id}")

task6 = ee.batch.Export.image.toDrive(
    image       = sift_img2.toFloat(),
    description = 'S2_SIFT_Date2',
    folder      = FOLDER,
    region      = AOI,
    scale       = SCALE,
    crs         = CRS,
    maxPixels   = MAX_PX
)
task6.start()
print(f"  ✔ Task started: S2_SIFT_Date2 | ID: {task6.id}")


# ═══════════════════════════════════════════════════════════════════════════════
#  SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════
print("\n" + "="*60)
print("ALL 6 EXPORT TASKS SUBMITTED")
print("="*60)
print("Monitor at: https://code.earthengine.google.com/tasks")
print()
print("Files to download from Google Drive → AgriSense/ folder:")
print("  S2_BOA_Punjab_Rabi2023.tif      → MAIN BOA product")
print("  S2_TOA_Punjab_Rabi2023.tif      → for DOS-1 benchmark")
print("  S2_CloudBands_Date1.tif         → for s2cloudless (10-band)")
print("  S2_CloudBands_Date2.tif         → for s2cloudless (10-band)")
print("  S2_SIFT_Date1.tif               → for SIFT (B4 single-band)")
print("  S2_SIFT_Date2.tif               → for SIFT (B4 single-band)")
print()
print("After download, place all files in same folder and run:")
print("  python task1_preprocess.py")

Preparing Export 1: S2 BOA Composite (MAIN PRODUCT)...
  ✔ Task started: S2_BOA_Punjab_Rabi2023 | ID: RVHLFECUFUGQID6BXZIUXVXA
Preparing Export 2: S2 TOA Composite (DOS-1 benchmark)...
  ✔ Task started: S2_TOA_Punjab_Rabi2023 | ID: OISZEKRHMZ4IMDVH3TDGEZDE
Preparing Exports 3 & 4: 10-band cloud-masking images (Date1 & Date2)...
  ✔ Task started: S2_CloudBands_Date1 | ID: 3UE5JN5GAZQDLUGNZAVZFUBT
  ✔ Task started: S2_CloudBands_Date2 | ID: UVMNU7OFEUHGFQY6XLVRFFDV
Preparing Exports 5 & 6: Single-band B4 images for SIFT...
  ✔ Task started: S2_SIFT_Date1 | ID: 3HOP7FQZMMCCHG7JFLK7LOW2
  ✔ Task started: S2_SIFT_Date2 | ID: JBMT4UEYXVNHA2VL6Y3SZY2J

ALL 6 EXPORT TASKS SUBMITTED
Monitor at: https://code.earthengine.google.com/tasks

Files to download from Google Drive → AgriSense/ folder:
  S2_BOA_Punjab_Rabi2023.tif      → MAIN BOA product
  S2_TOA_Punjab_Rabi2023.tif      → for DOS-1 benchmark
  S2_CloudBands_Date1.tif         → for s2cloudless (10-band)
  S2_CloudBands_Date2.tif         

In [ ]:

import ee
import geemap

# ─── GEE Initialisation ──────────────────────────────────────────────────────
ee.Authenticate()
ee.Initialize(project='dip-project-52918')   # ← replace with your project ID

# ─── Study Area & Parameters ─────────────────────────────────────────────────
AOI    = ee.Geometry.Rectangle([73.0, 30.5, 73.5, 31.0])
CRS    = 'EPSG:32642'
SCALE  = 10        # metres
FOLDER = 'AgriSense'
MAX_PX = 1e10

# Season window
SEASON_START = '2023-10-01'
SEASON_END   = '2024-04-30'

# Single-date windows for SIFT (15-day gap, stable phenology)
# ── WHY DECEMBER? ───────────────────────────────────────────────────────────
# Rabi wheat in Punjab is planted Oct–Nov and reaches early tillering by Dec.
# The canopy is uniform and changes slowly → SIFT finds stable texture features.
# A 30-day gap keeps reflectance change < 5% in most bands → RMSE < 1 px.
# ────────────────────────────────────────────────────────────────────────────
DATE1_START = '2023-12-01'
DATE1_END   = '2023-12-15'    # picks best single image in this 15-day window

DATE2_START = '2023-12-16'
DATE2_END   = '2023-12-31'    # picks best single image in second 15-day window

# 7 spectral bands for BOA/TOA composite products
COMPOSITE_BANDS = ['B2', 'B3', 'B4', 'B5', 'B8', 'B11', 'B12']

# 10 bands required by s2cloudless (exact order is mandatory)
# [B01, B02, B04, B05, B08, B8A, B09, B10, B11, B12]
CLOUD_BANDS = ['B1', 'B2', 'B4', 'B5', 'B8', 'B8A', 'B9', 'B10', 'B11', 'B12', 'QA60']
def get_best_image(collection, start, end, bands, aoi):
    """Return the least-cloudy single image in [start, end] clipped to aoi."""
    img = (collection
           .filterDate(start, end)
           .sort('CLOUDY_PIXEL_PERCENTAGE')
           .first()
           .select(bands)
           .clip(aoi))
    return img

print("Preparing Exports 3 & 4: 10-band cloud-masking images (Date1 & Date2)...")

s2_sr_all = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
               .filterBounds(AOI)
               .filterDate(DATE1_START, DATE2_END)
               .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

cloud_img1 = get_best_image(s2_sr_all, DATE1_START, DATE1_END, ['B1', 'B2', 'B4', 'B5', 'B8', 'B8A', 'B9', 'B11', 'B12', 'QA60'], AOI)
cloud_img2 = get_best_image(s2_sr_all, DATE2_START, DATE2_END, ['B1', 'B2', 'B4', 'B5', 'B8', 'B8A', 'B9','B11', 'B12', 'QA60'], AOI)

task3 = ee.batch.Export.image.toDrive(
    image       = cloud_img1.toFloat(),
    description = 'S2_CloudBands_Date1',
    folder      = FOLDER,
    region      = AOI,
    scale       = SCALE,
    crs         = CRS,
    maxPixels   = MAX_PX
)
task3.start()
print(f"  ✔ Task started: S2_CloudBands_Date1 | ID: {task3.id}")

task4 = ee.batch.Export.image.toDrive(
    image       = cloud_img2.toFloat(),
    description = 'S2_CloudBands_Date2',
    folder      = FOLDER,
    region      = AOI,
    scale       = SCALE,
    crs         = CRS,
    maxPixels   = MAX_PX
)
task4.start()
print(f"  ✔ Task started: S2_CloudBands_Date2 | ID: {task4.id}")

Preparing Exports 3 & 4: 10-band cloud-masking images (Date1 & Date2)...
  ✔ Task started: S2_CloudBands_Date1 | ID: CJ4QQVVYPZLAF5W5PIBCE57Y
  ✔ Task started: S2_CloudBands_Date2 | ID: AOZPBBQ67YMSTC7AUGUXSY65


In [ ]:
import rasterio
import numpy as np

with rasterio.open("/content/drive/MyDrive/AgriSense/S2_BOA_Punjab_Rabi2023.tif") as src:
    arr = src.read()

print(arr.min())
print(arr.max())
print(np.nanpercentile(arr, [1,5,50,95,99]))

nan
nan
[ 434.   561.5 1376.  3063.  3545. ]


In [ ]:
with rasterio.open("/content/drive/MyDrive/AgriSense/S2_BOA_Punjab_Rabi2023.tif") as src:
    boa_array   = src.read().astype(np.float32) * 0.0001

In [ ]:
print("nanmin:", np.nanmin(boa_array))
print("nanmax:", np.nanmax(boa_array))

for i, b in enumerate(['B2','B3','B4','B5','B8','B11','B12']):
    print(
        b,
        np.nanpercentile(boa_array[i], 2),
        np.nanpercentile(boa_array[i], 98)
    )

nanmin: 0.01675
nanmax: 0.8314
B2 0.03915 0.141
B3 0.0638 0.1767
B4 0.051799998 0.2109
B5 0.0954 0.2322
B8 0.19659999 0.38615
B11 0.16104999 0.3144
B12 0.09805 0.2845


In [ ]:
with rasterio.open("/content/drive/MyDrive/AgriSense/S2_CloudBands_Date1.tif") as src:
    print(src.count)
    print(src.descriptions)

10
('B1', 'B2', 'B4', 'B5', 'B8', 'B8A', 'B9', 'B11', 'B12', 'QA60')
